In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 환경변수 로드

# API 키와 환경명 가져오기
pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [3]:
# OpenAI 임베딩 모델 설정 (text-embedding-3-small 사용)
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
  model="text-embedding-3-small",
  openai_api_key=os.getenv("OPENAI_API_KEY")
)


In [5]:
from pinecone import Pinecone, ServerlessSpec

pinecone = Pinecone(
  api_key=pinecone_api_key
)
pinecone.list_indexes()

[
    {
        "name": "movie-index",
        "metric": "cosine",
        "host": "movie-index-dob4bs2.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "wiki",
        "metric": "cosine",
        "host": "wiki-dob4bs2.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "mov

In [6]:
pinecone.create_index(
  name="wiki",
  dimension=1536,
  metric="cosine",
  spec=ServerlessSpec(cloud="aws",
                      region="us-east-1"))

{
    "name": "wiki",
    "metric": "cosine",
    "host": "wiki-dob4bs2.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

In [7]:
%pip install datasets

  Using cached datasets-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached pyarrow-20.0.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (3.3 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.5.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (12 kB)
  Using cached multiprocess-0.70.16-py312-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached datasets-3.6.0-py3-none-any.whl (491 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
Using cached multiprocess-0.70.16-py312-none-any.whl (146 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 913.5 kB/s eta 0:00:000:00:010m
Using cached pyarrow-20.0.0-cp312-cp312-macosx_12_0_arm64.whl (30.8 MB)
Using cached filelock-3.18.0-py3-none-any.whl (16 kB)
Using cached xxhash-3.5.0-cp312-cp312-macosx_11_0_arm64.whl (30 kB)
   ━━━━━━

In [8]:
from datasets import load_dataset
# 위키피디아 데이터셋 로드
dataset = load_dataset("wikipedia", "20220301.simple", split="train[:100]",trust_remote_code=True)

Using the latest cached version of the module from /Users/sunny/.cache/huggingface/modules/datasets_modules/datasets/wikipedia/d41137e149b2ea90eead07e7e3f805119a8c22dd1d5b61651af8e3e3ee736001 (last modified on Sat Dec 21 15:48:42 2024) since it couldn't be found locally at wikipedia, or remotely on the Hugging Face Hub.


In [9]:
for record in dataset:
    # 각 레코드의 텍스트를 임베딩
    print(record)

{'id': '1', 'url': 'https://simple.wikipedia.org/wiki/April', 'title': 'April', 'text': 'April is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.\n\nApril always begins on the same day of week as July, and additionally, January in leap years. April always ends on the same day of the week as December.\n\nApril\'s flowers are the Sweet Pea and Daisy. Its birthstone is the diamond. The meaning of the diamond is innocence.\n\nThe Month \n\nApril comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.\n\nApril begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other\'s last days are exactly 35 weeks (245 days) apart.\n\nIn commo

In [11]:
len(dataset)

100

In [10]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
# 텍스트 분할기 설정
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=20,
    length_function=len,
    separators=["\n\n", "\n", " ", ""], # 문단 단위로 분활가능
)

In [12]:
dataset[0].keys()

dict_keys(['id', 'url', 'title', 'text'])

In [13]:
wiki_index = pinecone.Index("wiki")

In [14]:
texts = []
metas = []
batch_size = 100
count = 0
for i, sample in enumerate(dataset):
    text = sample["text"]
    metadata = {
        "title": sample["title"],
        "wiki_id": sample["id"],
        "url": sample["url"],
    }
    chunks = splitter.split_text(text)
    for i,chunk in enumerate(chunks):
        record = {
            "chunk_id": i,
            "text": chunk,
            **metadata
        }
        texts.append(chunk)
        metas.append(record)
        count += 1
        if count % batch_size == 0:
          vectors = embeddings.embed_documents(texts)
          ids = [f"{record['wiki_id']}-{record['chunk_id']}" for record in metas]
          wiki_index.upsert(zip(ids, vectors, metas))
          texts = []
          metas = []
          print(f"Upserted {count} records")

Upserted 100 records
Upserted 200 records
Upserted 300 records
Upserted 400 records
Upserted 500 records
Upserted 600 records
Upserted 700 records
Upserted 800 records
Upserted 900 records
Upserted 1000 records
Upserted 1100 records
Upserted 1200 records
Upserted 1300 records
Upserted 1400 records
Upserted 1500 records


In [16]:
from langchain_pinecone import PineconeVectorStore
vectorstore = PineconeVectorStore(
    index=wiki_index,
    embedding=embeddings,
    text_key="text"
)

questions = "벨기에(Belgium)는 어디 있나요?"
docs = vectorstore.similarity_search(questions, k=3)
for doc in docs:
    print(doc.metadata)

{'chunk_id': 0.0, 'title': 'Belgium', 'url': 'https://simple.wikipedia.org/wiki/Belgium', 'wiki_id': '103'}
{'chunk_id': 20.0, 'title': 'Belgium', 'url': 'https://simple.wikipedia.org/wiki/Belgium', 'wiki_id': '103'}
{'chunk_id': 2.0, 'title': 'Belgium', 'url': 'https://simple.wikipedia.org/wiki/Belgium', 'wiki_id': '103'}
